# Parameters
### Extract file_date parameter from Workflow

In [0]:
# Workflow execution

from datetime import datetime

dbutils.widgets.text(
    "file_date",
    datetime.now().strftime("%Y%m%d")
)

file_date = dbutils.widgets.get("file_date")

print(f"Processing data for file_date = {file_date}")

In [0]:
# Manual execution

#file_date = "20260207"

#print(f"Processing data for file_date = {file_date}")

### Remove Existing Data for the Current file_date

In [0]:
bronze_tables = [
    "online_customers",
    "offline_customers",
    "online_products",
    "offline_products",
    "online_orders",
    "offline_orders"
]

for table in bronze_tables:
    table_name = f"`namastesql-databricks-catalog`.bronze.{table}"

    if spark.catalog.tableExists(table_name):
        spark.sql(f"""
            DELETE FROM {table_name}
            WHERE file_date = '{file_date}'
        """)

# Bronze Layer
### Ingest Online and Offline Source Files for the Provided file_date

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Load Online Raw Files

In [0]:
# online orders table

df = spark.read.format("csv").option("header",True).load(
    f"s3://namaste-databricks/orders/online/{file_date}/online_orders.csv"
)

df_audit = df.withColumn("file_date", lit(file_date)) \
             .withColumn("source", lit("online")) \
             .withColumn("ingest_ts", current_timestamp())



df_audit.write.mode("append").saveAsTable(
    "`namastesql-databricks-catalog`.bronze.online_orders"
)


In [0]:
# online customers table

df = spark.read.format("csv").option("header",True).load(
    f"s3://namaste-databricks/orders/online/{file_date}/online_customers.csv"
)

df_audit = df.withColumn("file_date", lit(file_date)) \
             .withColumn("source", lit("online")) \
             .withColumn("ingest_ts", current_timestamp())

df_audit.write.mode("append").saveAsTable(
    "`namastesql-databricks-catalog`.bronze.online_customers"
)

In [0]:
# online products table

df = spark.read.format("csv").option("header",True).load(
    f"s3://namaste-databricks/orders/online/{file_date}/online_products.csv"
)

df_audit = df.withColumn("file_date", lit(file_date)) \
             .withColumn("source", lit("online")) \
             .withColumn("ingest_ts", current_timestamp())

df_audit.write.mode("append").saveAsTable(
    "`namastesql-databricks-catalog`.bronze.online_products"
)

### Load Offline Raw Files

In [0]:
# offline orders table

df = spark.read.format("csv").option("header",True).load(
    f"s3://namaste-databricks/orders/offline/{file_date}/offline_orders.csv"
)

df_audit = df.withColumn("file_date", lit(file_date)) \
             .withColumn("source", lit("offline")) \
             .withColumn("ingest_ts", current_timestamp())



df_audit.write.mode("append").saveAsTable(
    "`namastesql-databricks-catalog`.bronze.offline_orders"
)

In [0]:
# offline customers table

df = spark.read.format("csv").option("header",True).load(
    f"s3://namaste-databricks/orders/offline/{file_date}/offline_customers.csv"
)

df_audit = df.withColumn("file_date", lit(file_date)) \
             .withColumn("source", lit("offline")) \
             .withColumn("ingest_ts", current_timestamp())



df_audit.write.mode("append").saveAsTable(
    "`namastesql-databricks-catalog`.bronze.offline_customers"
)

In [0]:
# offline products table

df = spark.read.format("csv").option("header",True).load(
    f"s3://namaste-databricks/orders/offline/{file_date}/offline_products.csv"
)

df_audit = df.withColumn("file_date", lit(file_date)) \
             .withColumn("source", lit("offline")) \
             .withColumn("ingest_ts", current_timestamp())



df_audit.write.mode("append").saveAsTable(
    "`namastesql-databricks-catalog`.bronze.offline_products"
)